In [ ]:
!pip uninstall -y transformers
!pip install "transformers==4.50.0"

Found existing installation: transformers 5.16.1
Uninstalling transformers-5.16.1:
  Successfully uninstalled transformers-5.16.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 94.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.28.0
    Uninstalling huggingface_hub-1.28.0:
      Successfully uninstalled huggingface_hub-1.28.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.1
    Uninstalling tokenizers-0.23.1:
      Successfully uninstalled tokenizers-0.23.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 whic

In [ ]:
import transformers
print(transformers.__version__)

4.50.0


DistilBERT

In [4]:
import pandas as pd
import numpy as np
import tf_keras as keras
import tensorflow as tf
from transformers import DistilBertTokenizer,TFDistilBertForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,classification_report
from sklearn.preprocessing import OneHotEncoder

In [5]:
df = pd.read_csv(r"/content/sample_data/IMDB Dataset.csv")

print(df.head(5))

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [6]:
oe = OneHotEncoder(drop="first",sparse_output=False)
y = oe.fit_transform(df[["sentiment"]])

X = df["review"]

X_train_val,X_test,y_train_val,y_test = train_test_split(
    X.tolist(),y.tolist(),train_size=5000,test_size=1000,random_state=42,stratify=y
)

X_train,X_val,y_train,y_val = train_test_split(
    X_train_val,y_train_val,test_size=500,random_state=42,stratify=y_train_val
)

In [7]:
Tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

In [8]:
train_encodings = dict(Tokenizer(X_train,padding=True,truncation=True,max_length=128,return_tensors="tf"))
val_encodings = dict(Tokenizer(X_val,padding=True,truncation=True,max_length=128,return_tensors="tf"))
test_encodings = dict(Tokenizer(X_test,padding=True,truncation=True,max_length=128,return_tensors="tf"))
train_label = tf.convert_to_tensor(y_train)
val_label = tf.convert_to_tensor(y_val)
test_label = tf.convert_to_tensor(y_test)

In [9]:
Model = TFDistilBertForSequenceClassification.from_pretrained("bert-base-uncased",num_labels=2)
optimizer = keras.optimizers.Adam(learning_rate=2e-5)
loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

You are using a model of type bert to instantiate a model of type distilbert. This is not supported for all configurations of models and can yield errors.


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['bert.encoder.layer.6.output.dense.weight', 'bert.encoder.layer.11.intermediate.dense.bias', 'bert.encoder.layer.8.attention.self.query.weight', 'bert.encoder.layer.3.intermediate.dense.bias', 'bert.encoder.layer.5.intermediate.dense.weight', 'bert.encoder.layer.6.attention.output.dense.weight', 'bert.encoder.layer.6.output.LayerNorm.bias', 'bert.encoder.layer.7.attention.output.LayerNorm.bias', 'bert.encoder.layer.9.output.dense.weight', 'bert.encoder.layer.5.attention.self.key.weight', 'bert.encoder.layer.6.intermediate.dense.bias', 'bert.encoder.layer.7.attention.self.query.bias', 'bert.encoder.layer.8.attention.self.key.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.10.attention.output.LayerNorm.bias', 'bert.embeddings.LayerNorm.weight', 'bert.encoder.layer.6.attention.output.dense.bias', 'bert.encoder.layer.1.attention.sel

In [10]:
Model.compile(optimizer=optimizer,loss=loss,metrics=["accuracy"])
history = Model.fit(train_encodings,train_label,validation_data=(val_encodings,val_label),epochs=10)

Epoch 1/10
141/141 [==============================] - 170s 849ms/step - loss: 0.7030 - accuracy: 0.5020 - val_loss: 0.6781 - val_accuracy: 0.6120
Epoch 2/10
141/141 [==============================] - 121s 858ms/step - loss: 0.5666 - accuracy: 0.7033 - val_loss: 0.5685 - val_accuracy: 0.7320
Epoch 3/10
141/141 [==============================] - 124s 877ms/step - loss: 0.3734 - accuracy: 0.8422 - val_loss: 0.5588 - val_accuracy: 0.7580
Epoch 4/10
141/141 [==============================] - 124s 878ms/step - loss: 0.2475 - accuracy: 0.9107 - val_loss: 0.6502 - val_accuracy: 0.7600
Epoch 5/10
141/141 [==============================] - 124s 881ms/step - loss: 0.1744 - accuracy: 0.9409 - val_loss: 0.6648 - val_accuracy: 0.7700
Epoch 6/10
141/141 [==============================] - 124s 882ms/step - loss: 0.0902 - accuracy: 0.9718 - val_loss: 0.8145 - val_accuracy: 0.7600
Epoch 7/10
141/141 [==============================] - 124s 879ms/step - loss: 0.0726 - accuracy: 0.9753 - val_loss: 0.8501 -

In [11]:
pred = Model.predict(test_encodings)
pred = pred.logits
probabilities = tf.nn.softmax(pred,axis=1)
predictions = tf.argmax(probabilities,axis=1)

32/32 [==============================] - 12s 277ms/step


In [12]:
accuracy = accuracy_score(predictions,test_label)
print("Accuracy score: ",accuracy)
report = classification_report(predictions,test_label)
print("Classification report: ")
print(report)

Accuracy score:  0.796
Classification report: 
              precision    recall  f1-score   support

           0       0.82      0.78      0.80       520
           1       0.78      0.81      0.79       480

    accuracy                           0.80      1000
   macro avg       0.80      0.80      0.80      1000
weighted avg       0.80      0.80      0.80      1000

